In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output

## 1. Q-learning in the wild (3 pts)

Here we use the qlearning agent on taxi env from openai gym.
You will need to insert a few agent functions here.

In [ ]:
import random,math
import numpy as np
from collections import defaultdict

class QLearningAgent():
  """
    Q-Learning Agent

    Instance variables you have access to
      - self.epsilon (exploration prob)
      - self.alpha (learning rate)
      - self.discount (discount rate aka gamma)

    Functions you should use
      - self.getLegalActions(state)
        which returns legal actions for a state
      - self.getQValue(state,action)
        which returns Q(state,action)
      - self.setQValue(state,action,value)
        which sets Q(state,action) := value

    !!!Important!!!
    NOTE: please avoid using self._qValues directly to make code cleaner
  """
  def __init__(self,alpha,epsilon,discount,getLegalActions):
    "We initialize agent and Q-values here."
    self.getLegalActions= getLegalActions
    self._qValues = defaultdict(lambda:defaultdict(lambda:0))
    self.alpha = alpha
    self.epsilon = epsilon
    self.discount = discount

  def getQValue(self, state, action):
    #print(state)
    #print(action)
    if not (state in self._qValues) or not (action in self._qValues[state]):
        return 0.0
    return self._qValues[state][action]

  def setQValue(self,state,action,value):
    """
      Sets the Qvalue for [state,action] to the given value
    """
    self._qValues[state][action] = value

#---------------------#start of your code#---------------------#

  def getValue(self, state):
    """
      Returns max_action Q(state,action)
      where the max is over legal actions.
    """
    possibleActions = self.getLegalActions(state)
    if len(possibleActions) == 0:
        return 0.0
    q_values = [self.getQValue(state, a) for a in possibleActions]
    return max(q_values)

  def getPolicy(self, state):
    """
      Compute the best action to take in a state.
    """
    possibleActions = self.getLegalActions(state)
    if len(possibleActions) == 0:
        return None
    q_values = [(self.getQValue(state, a), a) for a in possibleActions]
    return max(q_values)[1]

  def getAction(self, state):
    """
      Compute the action to take in the current state, including exploration.
      With probability self.epsilon, we should take a random action.
      otherwise - the best policy action (self.getPolicy).
    """
    possibleActions = self.getLegalActions(state)
    if len(possibleActions) == 0:
        return None

    if random.random() < self.epsilon:
        action = random.choice(possibleActions)
    else:
        action = self.getPolicy(state)

    return action

  def update(self, state, action, nextState, reward):
    """
      You should do your Q-Value update here
    """
    gamma = self.discount
    learning_rate = self.alpha

    current_q = self.getQValue(state, action)
    next_q = self.getValue(nextState)
    new_q = current_q + learning_rate * (reward + gamma * next_q - current_q)

    self.setQValue(state, action, new_q)

In [ ]:
import gym
env = gym.make("Taxi-v3")
n_actions = env.action_space.n

In [ ]:
def play_and_train(env,agent,t_max=10**4):
    """This function should
    - run a full game, actions given by agent.getAction(s)
    - train agent using agent.update(...) whenever possible
    - return total reward"""
    total_reward = 0.0
    s = env.reset()

    for t in range(t_max):
        a = agent.getAction(s)
        next_s,r,done,_ = env.step(a)
        agent.update(s, a, next_s, r)
        s = next_s
        total_reward +=r
        if done:
            break
    return total_reward

In [ ]:
agent = QLearningAgent(alpha=0.1, epsilon=0.1, discount=0.95,
                       getLegalActions = lambda s: range(n_actions))

Достигните положительной награды, постройте график

In [ ]:
from IPython.display import clear_output

In [ ]:
from IPython.display import clear_output
rewards = []
for i in range(1000):
    rewards.append(play_and_train(env,agent))
    agent.epsilon *= 0.995  # уменьшаем epsilon со временем

    if i % 100 ==0:
        clear_output(True)
        print(f"Episode: {i}, Epsilon: {agent.epsilon:.4f}")
        plt.plot(rewards)
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.title('Taxi-v3 Training Progress')
        plt.show()

## 3. Continuous state space (2 pt)

Чтобы использовать табличный q-learning на continuous состояниях, надо как-то их обрабатывать и бинаризовать. Придумайте способ разбивки на дискретные состояния.

In [ ]:
env = gym.make("CartPole-v0")
n_actions = env.action_space.n
print("first state:%s"%(env.reset()))

### Play a few games

Постройте распределения различных частей состояния игры. Сыграйте несколько игр и запишите все состояния.

In [ ]:
# Собираем состояния
states_collected = []
for _ in range(50):
    s = env.reset()
    done = False
    while not done:
        states_collected.append(s)
        a = env.action_space.sample()
        s, r, done, _ = env.step(a)

states_collected = np.array(states_collected)
print("State ranges:")
print(f"Position: [{states_collected[:,0].min():.3f}, {states_collected[:,0].max():.3f}]")
print(f"Velocity: [{states_collected[:,1].min():.3f}, {states_collected[:,1].max():.3f}]")
print(f"Angle: [{states_collected[:,2].min():.3f}, {states_collected[:,2].max():.3f}]")
print(f"Angular velocity: [{states_collected[:,3].min():.3f}, {states_collected[:,3].max():.3f}]")

# Визуализация распределений
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, (ax, title) in enumerate(zip(axes.flat, ['Position', 'Velocity', 'Angle', 'Angular Velocity'])):
    ax.hist(states_collected[:, i], bins=30, alpha=0.7)
    ax.set_title(f'Distribution of {title}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## Binarize environment

In [ ]:
from gym.core import ObservationWrapper
class Binarizer(ObservationWrapper):

    def to_bin(self, value, bins):
        return np.digitize(value, bins)

    def _observation(self,state):
        # Определяем границы бинов на основе собранных данных
        bins = [
            np.linspace(-2.4, 2.4, 10),   # position
            np.linspace(-3.0, 3.0, 10),   # velocity
            np.linspace(-0.2, 0.2, 10),   # angle
            np.linspace(-3.0, 3.0, 10)    # angular velocity
        ]

        state = (self.to_bin(state[0], bins[0]),
                 self.to_bin(state[1], bins[1]),
                 self.to_bin(state[2], bins[2]),
                 self.to_bin(state[3], bins[3]))
        return state

In [ ]:
env = Binarizer(gym.make("CartPole-v0"))
print("Binarized state example:", env.reset())

## Learn

In [ ]:
agent = QLearningAgent(alpha=0.1, epsilon=0.3, discount=0.98,
                       getLegalActions = lambda s: range(n_actions))

In [ ]:
rewards = []
rewBuf = []
for i in range(10000):
    for j in range(100):
        rewards.append(play_and_train(env,agent))
    agent.epsilon *= 0.995
    rewBuf.append(np.mean(rewards[-100:]))

    if i % 100 == 0:
        clear_output(True)
        print(f"Iteration: {i}, Epsilon: {agent.epsilon:.4f}")
        print(f"Average reward (last 100): {rewBuf[-1]:.2f}")
        plt.plot(rewBuf)
        plt.xlabel('Iteration (x100 episodes)')
        plt.ylabel('Average Reward')
        plt.title('CartPole Training Progress')

        if rewBuf[-1] > 195:
            print("Win!")
            break
        plt.show()

## 4. Experience replay (5 pts)

In [ ]:
import random
class ReplayBuffer(object):
    def __init__(self, size):
        """Create Replay buffer.
        Parameters
        ----------
        size: int
            Max number of transitions to store in the buffer. When the buffer
            overflows the old memories are dropped.
        """
        self._storage = []
        self._maxsize = size
        self._replaceId = 0

    def __len__(self):
        return len(self._storage)

    def add(self, obs_t, action, reward, obs_tp1, done):
        '''
        Make sure, _storage will not exceed _maxsize.
        '''
        data = (obs_t, action, reward, obs_tp1, done)
        if len(self._storage) == self._maxsize:
            self._storage[self._replaceId] = data
            self._replaceId = (self._replaceId + 1) % self._maxsize
        else:
            self._storage.append(data)

    def sample(self, batch_size):
        """Sample a batch of experiences.
        Parameters
        ----------
        batch_size: int
            How many transitions to sample.
        Returns
        -------
        obs_batch: np.array
            batch of observations
        act_batch: np.array
            batch of actions executed given obs_batch
        rew_batch: np.array
            rewards received as results of executing act_batch
        next_obs_batch: np.array
            next set of observations seen after executing act_batch
        done_mask: np.array
            done_mask[i] = 1 if executing act_batch[i] resulted in
            the end of an episode and 0 otherwise.
        """
        indices = np.random.choice(len(self._storage), batch_size)
        states, actions, rewards, next_states, dones = [], [], [], [], []

        for idx in indices:
            s, a, r, ns, d = self._storage[idx]
            states.append(s)
            actions.append(a)
            rewards.append(r)
            next_states.append(ns)
            dones.append(d)

        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

Some tests to make sure your buffer works right

In [ ]:
import numpy as np
replay = ReplayBuffer(2)
obj1 = tuple(range(5))
obj2 = tuple(range(5, 10))
replay.add(*obj1)

# Для проверки сэмплирования
sample = replay.sample(1)
assert all(sample[0][0] == obj1[0]), "If there's just one object in buffer, it must be retrieved by buf.sample(1)"

replay.add(*obj2)
assert len(replay._storage)==2, "Please make sure __len__ methods works as intended."
replay.add(*obj2)
assert len(replay._storage)==2, "When buffer is at max capacity, replace objects instead of adding new ones."

print ("Success!")

Now let's use this buffer to improve training:

In [ ]:
import gym
env = Binarizer(gym.make('CartPole-v0'))
n_actions = env.action_space.n

In [ ]:
agent = QLearningAgent(alpha=0.1, epsilon=0.3, discount=0.98,
                       getLegalActions = lambda s: range(n_actions))
replay = ReplayBuffer(10000)

In [ ]:
def play_and_train_with_replay(env, agent, replay_buffer, t_max=10**4, batch_size=10):
    """This function should
    - run a full game, actions given by agent.getAction(s)
    - train agent using agent.update(...) whenever possible
    - return total reward"""
    total_reward = 0.0
    s = env.reset()

    for t in range(t_max):
        action = agent.getAction(s)
        next_s, r, done, _ = env.step(action)

        # заполняем реплей буфер
        replay_buffer.add(s, action, r, next_s, done)

        # онлайн обучение
        agent.update(s, action, next_s, r)

        # обучение из буфера
        if len(replay_buffer) >= batch_size:
            states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
            for i in range(batch_size):
                agent.update(states[i], actions[i], next_states[i], rewards[i])

        s = next_s
        total_reward += r
        if done:
            break

    return total_reward

Train with experience replay

In [ ]:
rewards = []
rewBuf = []
for i in range(10000):
    for j in range(100):
        rewards.append(play_and_train_with_replay(env,agent, replay, batch_size=32))
    agent.epsilon *= 0.995
    rewBuf.append(np.mean(rewards[-100:]))

    if i % 100 == 0:
        clear_output(True)
        print(f"Iteration: {i}, Epsilon: {agent.epsilon:.4f}")
        print(f"Average reward (last 100): {rewBuf[-1]:.2f}")
        plt.plot(rewBuf)
        plt.xlabel('Iteration (x100 episodes)')
        plt.ylabel('Average Reward')
        plt.title('CartPole Training with Experience Replay')

        if rewBuf[-1] > 195:
            print("Win!")
            break
        plt.show()